In [2]:
import pandas as pd
import json
from pathlib import Path

In [3]:
gold = Path("GOLD_STD_LABELS.json")
o = json.loads(gold.read_text(encoding="utf8"))

In [5]:
def simplify(item):
    return {
        "id"      : item["data"]["id"],
        "project" : item["data"]["project"],
        "file_id" : item["data"]["file_id"],
        "language": item["data"]["language"],
        "text"    : item["data"]["text"],
        "annotations": [ r["value"] for a in item["annotations"] for r in a["result"]]
    }

#simplify(o[0])

In [29]:
from transformers import AutoTokenizer
import numpy as np

tok         = AutoTokenizer.from_pretrained("xaviergillard/xlm-roberta-large-vieille-france")
category2id = { 'O': 0, 'B-PERS': 1, 'I-PERS': 2, 'B-LOC': 3, 'I-LOC': 4, 'B-DATE': 5, 'I-DATE': 6, 'IGN': -100 }
id2category = {v:k for k,v in category2id.items()}

def as_dataset_entry(entry):
    simp  = simplify(entry)
    annot = sorted(simp["annotations"], key=lambda a: (a["start"], a["end"]))
    enc   = tok(
        simp["text"], 
        max_length = 510,
        padding = True,
        truncation = True,
        stride = 0,
        return_overflowing_tokens = True,
        add_special_tokens = True,
        return_offsets_mapping = True,
        is_split_into_words=False,
        return_special_tokens_mask = True,
        return_tensors='np',
    )
    
    tokshape= enc["input_ids"].shape
    tokens  = enc["input_ids"].flatten()
    ignore  = ((~(enc["attention_mask"]).astype(np.bool)) | enc["special_tokens_mask"].astype(np.bool)).flatten()
    labels  = np.zeros_like(tokens)
    offsets = enc["offset_mapping"].reshape((-1, 2))
    
    ## on aligne pour du vrai
    iob = 'O'
    tpos= 0
    apos= 0
    
    while apos < len(annot) and tpos < len(tokens):
        if ignore[tpos]:
            labels[tpos] = category2id["IGN"]
            tpos += 1
            continue
        
        # retrouve le début et la fin des segments
        astart,aend = annot[apos]["start"],annot[apos]["end"]
        tstart,tend = offsets[tpos]
        atag        = annot[apos]["labels"][0]
        
        # retrouve le tag correspondant a une annotation
        match atag:
            case "PERSON": 
                atag = "PERS"
            case "LOCATION":
                atag = "LOC"
            case "DATE":
                atag = "DATE"
    
        # déplacements
        if tstart > aend:
            apos += 1
        elif astart > tend:
            tpos += 1
        else: # range overlap
            iob = "B-" if astart == tstart else "I-"
            atag= iob + atag
            labels[tpos] = category2id[atag]
            tpos += 1

    labels = labels.reshape(tokshape)
    return {
        "input_ids":           enc["input_ids"],
        "attention_mask":      enc["attention_mask"],
        "special_tokens_mask": enc["special_tokens_mask"],
        "labels":              labels
    }


In [39]:
entries = [ as_dataset_entry(x) for x in o ]
dataset = pd.DataFrame(entries).explode(["input_ids", "attention_mask", "special_tokens_mask", "labels"]).reset_index()
dataset = dataset[["input_ids", "attention_mask", "special_tokens_mask", "labels"]]

In [40]:
dataset

,input_ids,attention_mask,special_tokens_mask,labels
0,"[0, 158003, 60100, 86399, 444, 2285, 156, 56, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-100, 0, 0, 0, 0, 0, 1, 2, 2, 0, 0, 0, 0, 0, ..."
1,"[0, 13, 2278, 700, 208, 33, 491, 293, 820, 113...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,"[0, 65487, 28117, 23, 60780, 7, 10517, 824, 64...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,"[0, 14799, 26301, 23, 48566, 17990, 134231, 13...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,"[0, 33, 4535, 33, 28117, 44578, 2594, 607, 211...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
2085,"[0, 1931, 429, 4369, 13283, 18, 22, 10513, 970...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2086,"[0, 122, 13283, 7, 5, 1260, 33, 6, 47704, 607,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2087,"[0, 83, 61636, 76456, 24099, 435, 720, 13283, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2088,"[0, 28117, 171682, 390, 47112, 23, 293, 642, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [41]:
dataset.to_parquet("gold_standard.parquet")